In [ ]:
import napari
from tifffile import imread
import pandas as pd
import numpy as np

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Load image

image = imread('path_to_image_of_choice')

In [ ]:
print(image.dtype, image.shape, np.min(image), np.max(image))

In [ ]:
# Loads image mask

image_mask = imread('path_to_mask_of_choice')

In [ ]:
print(image_mask .dtype, image_mask .shape, np.min(image_mask ), np.max(image_mask ))

In [ ]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [ ]:
# Adds image to Napari viewer

viewer.add_image(image)

In [ ]:
# Adds mask image to Napari viewer

viewer.add_image(image_mask,
                 colormap="gist_earth",
                 contrast_limits=[0,8],
                 opacity=0.2)

In [ ]:
# Loads detected spots

spots_path = r"path_to_spots_file_of_choice"
spots = pd.read_csv(spots_path)
spots = spots.loc[:, ~spots.columns.str.startswith('Unnamed')]
spots.head()

In [ ]:
# Creates df that contains xyct axes in the same order as image --> tcxy

spots_tcyxcoord = spots.iloc[:, np.r_[0,3,2]]
spots_tcyxcoord.insert(1, "channel", 0)
spots_tcyxcoord

In [ ]:
# Creates array from df (necessary to use in the points layer of Napari)

spots_tcyxarray = spots_tcyxcoord.to_numpy()
spots_tcyxarray

In [ ]:
# Adds detected spots to points layer

viewer.add_points(spots_tcyxcoord,
                  face_color='#ffffff00',
                  edge_color='magenta',
                  name='spots_log')

In [ ]:
# Loads tracks

tracks_path = r"path_to_tracks_file_of_choice"
tracks = pd.read_csv(tracks_path)
tracks = tracks.loc[:, ~tracks.columns.str.startswith('Unnamed')]
tracks.head()

In [ ]:
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

tracks_vis = tracks.iloc[:, np.r_[13,0,3,2]].sort_values(["unique_id", "frame"])
tracks_vis.insert(2, "channel", 0)
tracks_vis.head(20)

In [ ]:
viewer.add_tracks(tracks_vis[['unique_id', 'frame', 'channel', 'y', 'x']], name='tracks', 
                  properties={
                      'id': tracks_vis['unique_id'].values,
                             },
                  color_by='id',
                  colormap='hsv', 
                 )